# Module 9 — Inequality and Heterogeneous Agents
## When Identical People Face Different Luck: Wealth, Distribution, and Piketty

**From Smith to Simulation: Computing the Ideas that Built Economics**  
The University of Edinburgh · School of Economics

---

## Part I — The History

### The Return of Inequality

For most of the twentieth century, mainstream economics had surprisingly little to say about inequality. The profession was busy with other things — growth models, business cycles, game theory — and the prevailing view, following **Simon Kuznets** (1955), was that inequality would take care of itself. As countries industrialised, inequality would first rise and then fall, tracing an inverted U-shape. The data seemed to support this: income inequality in the United States and Europe declined steadily from the 1930s through the 1970s.

Then something changed. Starting around 1980, inequality began rising sharply in the English-speaking world. The top 1% income share in the United States, which had fallen from about 24% in 1928 to under 9% in the 1970s, climbed back to over 20% by 2012. The comfortable Kuznets narrative broke down.

In 2014, a French economist published a 700-page book that became an unlikely global bestseller. **Thomas Piketty's** *Capital in the Twenty-First Century* marshalled two centuries of tax data from over twenty countries to tell a dramatic story: the mid-twentieth-century decline in inequality was the exception, not the rule. The natural tendency of capitalism, Piketty argued, is toward *increasing* concentration of wealth.

> *"When the rate of return on capital exceeds the rate of growth of output and income, as it did in the nineteenth century and seems quite likely to do again in the twenty-first, capitalism automatically generates arbitrary and unsustainable inequalities that radically undermine the meritocratic values on which democratic societies are based."*  
> — Thomas Piketty, *Capital in the Twenty-First Century* (2014)

### The Historical Arc of Inequality

Piketty and his collaborators — notably **Emmanuel Saez** and **Gabriel Zucman** — documented three distinct eras:

1. **The Gilded Age (1870–1914).** Extreme concentration of wealth in the hands of industrial magnates — the Rockefellers, Carnegies, and Vanderbilts. The top 10% owned over 80% of total wealth in both Europe and America.

2. **The Great Compression (1930–1975).** Two world wars, the Great Depression, progressive taxation, and the rise of the welfare state dramatically compressed the wealth distribution. This was the anomaly that Kuznets mistook for an iron law.

3. **The Great Divergence (1980–present).** Tax cuts, financial deregulation, globalisation, and skill-biased technological change reversed the compression. Wealth concentration approached Gilded Age levels.

Piketty's central formula was deceptively simple: **r > g**. When the rate of return on capital ($r$) exceeds the growth rate of the economy ($g$), inherited wealth grows faster than earned income, and inequality widens inexorably. The wars and policy interventions of the twentieth century temporarily pushed $r$ below $g$, but that era may be ending.

### The Aiyagari Model: Inequality from Identical People

Even before Piketty's empirical revolution, theorists had been asking: can we explain inequality without assuming that people are fundamentally different? In 1994, **S. Rao Aiyagari** published a landmark paper showing that the answer is yes.

In the **Aiyagari model**, all agents are *identical* — same preferences, same abilities, same initial conditions. The only difference is *luck*: each period, agents receive random income shocks (modelled as a Markov chain). Some are lucky for several periods in a row and accumulate wealth; others face bad luck and deplete their savings. Even though everyone is identical *ex ante*, the wealth distribution that emerges in the long run is highly unequal and right-skewed — a few agents become very rich while many remain poor.

This was a profound insight: **inequality is an emergent property** of an economy with incomplete markets and idiosyncratic risk, not necessarily the result of differences in talent, effort, or inheritance.

### Edinburgh Connection: Angus Deaton and "The Great Escape"

**Angus Deaton** (1945–2025), born in Edinburgh and educated at Fettes College and then the University of Cambridge, won the Nobel Prize in Economics in 2015 — one year after Piketty's book appeared. Deaton's life work centred on how to *measure* living standards and inequality, particularly in the developing world.

His book *The Great Escape: Health, Wealth, and the Origins of Inequality* (2013) told a story both hopeful and cautionary. On one hand, humanity has made extraordinary progress: billions have escaped poverty, life expectancy has doubled, and material living standards have risen spectacularly. On the other hand, this progress has been deeply uneven — both across and within countries. Deaton warned that extreme inequality can undermine the very institutions that make growth possible.

Deaton's Scottish roots show in his work: the same empirical rigour and moral seriousness that characterised the Scottish Enlightenment — the insistence that economics must grapple with real data about real lives, not just elegant abstractions. His development of household survey methods and consumption-based measures of welfare gave economists the tools to see inequality where GDP averages hid it.

---

## Part II — The Computation

### Setting Up

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)  # for reproducibility

### Income as a Markov Chain

In the Aiyagari model, each agent's income follows a **Markov chain** — a stochastic process where next period's income depends only on this period's income, not on the full history. We define three income states:

| State | Label | Income ($y$) |
|-------|-------|--------------|
| 0 | Low | 0.5 |
| 1 | Medium | 1.0 |
| 2 | High | 2.0 |

The **transition matrix** $P$ specifies the probability of moving between states. Entry $P_{ij}$ is the probability of moving from state $i$ to state $j$. Each row sums to 1.

We choose a matrix with moderate persistence — agents tend to stay in their current state but can move up or down.

In [ ]:
# Income levels for three states
income_levels = np.array([0.5, 1.0, 2.0])

# Transition matrix: P[i, j] = Prob(state j next period | state i this period)
P = np.array([
    [0.6, 0.3, 0.1],   # from Low:    60% stay, 30% -> Med, 10% -> High
    [0.2, 0.6, 0.2],   # from Medium: 20% -> Low, 60% stay, 20% -> High
    [0.1, 0.3, 0.6],   # from High:   10% -> Low, 30% -> Med, 60% stay
])

# Verify rows sum to 1
print("Transition matrix P:")
print(P)
print(f"\nRow sums: {P.sum(axis=1)}")
print("\nInterpretation: a low-income agent has a 60% chance of staying low,")
print("a 30% chance of moving to medium, and a 10% chance of jumping to high income.")

### The Stationary Distribution of Income

A Markov chain has a **stationary distribution** $\pi$ such that $\pi P = \pi$. This tells us, in the long run, what fraction of the population is in each income state. We find it by computing the left eigenvector of $P$ with eigenvalue 1.

In [ ]:
# Find stationary distribution by solving pi @ P = pi
# Equivalently, pi is the left eigenvector of P with eigenvalue 1
eigenvalues, eigenvectors = np.linalg.eig(P.T)

# Find the eigenvector corresponding to eigenvalue 1
idx = np.argmin(np.abs(eigenvalues - 1.0))
pi = np.real(eigenvectors[:, idx])
pi = pi / pi.sum()  # normalise to sum to 1

print("Stationary distribution of income:")
for i, (state, prob) in enumerate(zip(['Low', 'Medium', 'High'], pi)):
    print(f"  {state} (y = {income_levels[i]}): {prob:.4f} ({prob*100:.1f}%)")

mean_income = np.dot(pi, income_levels)
print(f"\nLong-run mean income: {mean_income:.4f}")

### Simulating Income Paths for Many Agents

Now we simulate the Markov chain forward for a large number of agents. Each agent starts in a randomly assigned state and transitions according to $P$ each period.

In [ ]:
def simulate_markov(P, n_agents, n_periods, seed=42):
    """Simulate a Markov chain for many agents over many periods.
    
    Returns an array of shape (n_agents, n_periods) with state indices.
    """
    rng = np.random.default_rng(seed)
    n_states = P.shape[0]
    states = np.zeros((n_agents, n_periods), dtype=int)
    
    # Initial state: draw from stationary distribution
    states[:, 0] = rng.choice(n_states, size=n_agents, p=pi)
    
    # Simulate transitions
    for t in range(1, n_periods):
        u = rng.random(n_agents)
        for s in range(n_states):
            mask = (states[:, t-1] == s)
            cum_probs = np.cumsum(P[s])
            states[mask, t] = np.searchsorted(cum_probs, u[mask])
    
    return states

# Simulate
n_agents = 5000
n_periods = 200
state_paths = simulate_markov(P, n_agents, n_periods)
income_paths = income_levels[state_paths]  # convert state indices to income values

print(f"Simulated {n_agents} agents over {n_periods} periods.")
print(f"Income path shape: {income_paths.shape}")

# Plot a few sample income paths
fig, ax = plt.subplots(figsize=(12, 4))
for i in range(5):
    ax.step(range(50), income_paths[i, :50], alpha=0.7, linewidth=1.2, label=f'Agent {i+1}')
ax.set_xlabel('Period', fontsize=12)
ax.set_ylabel('Income', fontsize=12)
ax.set_title('Sample Income Paths (First 50 Periods)', fontsize=13)
ax.set_yticks(income_levels)
ax.set_yticklabels(['Low (0.5)', 'Med (1.0)', 'High (2.0)'])
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Each agent follows the same Markov chain — the only difference is luck.")

### From Income to Wealth: The Accumulation Equation

Agents accumulate wealth according to:

$$w_{t+1} = (1 + r) \cdot w_t + y_t - c_t$$

where:
- $w_t$ is wealth at time $t$
- $r$ is the interest rate (return on savings)
- $y_t$ is income at time $t$ (drawn from the Markov chain)
- $c_t$ is consumption at time $t$

We use a simple **consumption rule**: agents consume a fixed fraction of their available resources (wealth plus income):

$$c_t = \alpha \cdot \bigl((1+r) \cdot w_t + y_t\bigr)$$

where $\alpha \in (0, 1)$ is the consumption rate. We also impose a **borrowing constraint**: wealth cannot fall below zero ($w_t \geq 0$). This is the key friction in the Aiyagari model — agents cannot perfectly insure themselves against bad income shocks.

In [ ]:
def simulate_wealth(income_paths, r=0.02, alpha=0.9, w0=1.0):
    """Simulate wealth accumulation for all agents.
    
    Parameters:
        income_paths: array of shape (n_agents, n_periods) with income values
        r: interest rate (return on savings)
        alpha: consumption rate (fraction of resources consumed)
        w0: initial wealth for all agents
    
    Returns:
        wealth: array of shape (n_agents, n_periods)
    """
    n_agents, n_periods = income_paths.shape
    wealth = np.zeros((n_agents, n_periods))
    wealth[:, 0] = w0
    
    for t in range(n_periods - 1):
        resources = (1 + r) * wealth[:, t] + income_paths[:, t]
        consumption = alpha * resources
        wealth[:, t+1] = np.maximum(resources - consumption, 0.0)  # borrowing constraint
    
    return wealth

# Simulate wealth with baseline parameters
r = 0.02       # 2% interest rate
alpha = 0.9    # consume 90% of resources, save 10%
wealth = simulate_wealth(income_paths, r=r, alpha=alpha, w0=1.0)

print(f"Wealth simulation complete.")
print(f"Final period — Mean wealth: {wealth[:, -1].mean():.2f}")
print(f"Final period — Median wealth: {np.median(wealth[:, -1]):.2f}")
print(f"Final period — Max wealth: {wealth[:, -1].max():.2f}")
print(f"Final period — Min wealth: {wealth[:, -1].min():.2f}")
print(f"\nNote: mean > median indicates a right-skewed distribution.")

### The Stationary Wealth Distribution

After enough periods, the wealth distribution converges to a **stationary distribution**. Let's look at how wealth evolves over time and then examine the final distribution.

In [ ]:
# Plot evolution of wealth distribution over time
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, t in zip(axes, [10, 50, n_periods - 1]):
    ax.hist(wealth[:, t], bins=50, density=True, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_xlabel('Wealth', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title(f'Period {t}', fontsize=13)
    ax.axvline(np.mean(wealth[:, t]), color='red', linestyle='--', linewidth=1.5, label='Mean')
    ax.axvline(np.median(wealth[:, t]), color='orange', linestyle='--', linewidth=1.5, label='Median')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Evolution of the Wealth Distribution', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("The distribution is RIGHT-SKEWED: a long tail of wealthy agents.")
print("The mean exceeds the median — pulled up by a few very rich agents.")
print("This emerges from identical agents facing different luck — Aiyagari's insight.")

In [ ]:
# Detailed histogram of the final wealth distribution
final_wealth = wealth[:, -1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(final_wealth, bins=80, density=True, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(np.mean(final_wealth), color='red', linestyle='--', linewidth=2, label=f'Mean = {np.mean(final_wealth):.2f}')
ax.axvline(np.median(final_wealth), color='orange', linestyle='--', linewidth=2, label=f'Median = {np.median(final_wealth):.2f}')

# Mark the top 10% threshold
p90 = np.percentile(final_wealth, 90)
ax.axvline(p90, color='darkred', linestyle=':', linewidth=2, label=f'Top 10% threshold = {p90:.2f}')

ax.set_xlabel('Wealth', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Stationary Wealth Distribution (Final Period)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Wealth shares
total_w = final_wealth.sum()
sorted_w = np.sort(final_wealth)
top10_share = sorted_w[int(0.9 * n_agents):].sum() / total_w
top1_share = sorted_w[int(0.99 * n_agents):].sum() / total_w
bottom50_share = sorted_w[:int(0.5 * n_agents)].sum() / total_w

print(f"Wealth shares:")
print(f"  Top 1% own:    {top1_share*100:.1f}% of total wealth")
print(f"  Top 10% own:   {top10_share*100:.1f}% of total wealth")
print(f"  Bottom 50% own: {bottom50_share*100:.1f}% of total wealth")

### The Lorenz Curve

The **Lorenz curve** is the standard graphical tool for visualising inequality. It plots the cumulative share of wealth (y-axis) against the cumulative share of the population (x-axis), with agents sorted from poorest to richest.

- If wealth were perfectly equal, the Lorenz curve would be the 45-degree line (the "line of equality").
- The more unequal the distribution, the more the Lorenz curve bows below the 45-degree line.

In [ ]:
def lorenz_curve(wealth_array):
    """Compute the Lorenz curve from an array of wealth values.
    
    Returns:
        pop_share: cumulative population share (0 to 1)
        wealth_share: cumulative wealth share (0 to 1)
    """
    sorted_wealth = np.sort(wealth_array)
    n = len(sorted_wealth)
    cumulative_wealth = np.cumsum(sorted_wealth)
    total_wealth = sorted_wealth.sum()
    
    # Prepend zero for the origin
    pop_share = np.concatenate(([0], np.arange(1, n + 1) / n))
    wealth_share = np.concatenate(([0], cumulative_wealth / total_wealth))
    
    return pop_share, wealth_share

# Compute and plot the Lorenz curve
pop_share, wealth_share = lorenz_curve(final_wealth)

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Perfect equality')
ax.plot(pop_share, wealth_share, color='steelblue', linewidth=2.5, label='Lorenz curve')
ax.fill_between(pop_share, wealth_share, pop_share, color='steelblue', alpha=0.15)

ax.set_xlabel('Cumulative share of population (poorest to richest)', fontsize=12)
ax.set_ylabel('Cumulative share of wealth', fontsize=12)
ax.set_title('Lorenz Curve of the Simulated Wealth Distribution', fontsize=13)
ax.legend(fontsize=11, loc='upper left')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Annotate
ax.annotate('The shaded area between\nthe curves measures inequality',
            xy=(0.55, 0.35), fontsize=10, color='steelblue',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray'))

plt.tight_layout()
plt.show()

### The Gini Coefficient

The **Gini coefficient** is a single number that summarises the degree of inequality:

$$G = \frac{A}{A + B}$$

where $A$ is the area between the line of equality and the Lorenz curve, and $A + B$ is the total area under the line of equality (which equals 0.5).

Equivalently:

$$G = 1 - 2 \int_0^1 L(p) \, dp$$

where $L(p)$ is the Lorenz curve function.

- $G = 0$ means perfect equality (everyone has the same wealth).
- $G = 1$ means perfect inequality (one person has everything).
- Real-world Gini coefficients for wealth range from about 0.5 (Scandinavia) to 0.85 (the United States).

In [ ]:
def gini_coefficient(wealth_array):
    """Compute the Gini coefficient from an array of wealth values.
    
    Uses the trapezoidal approximation of the area under the Lorenz curve.
    """
    pop_share, wealth_share = lorenz_curve(wealth_array)
    # Area under Lorenz curve using the trapezoidal rule
    area_under_lorenz = np.trapz(wealth_share, pop_share)
    # Gini = 1 - 2 * (area under Lorenz curve)
    return 1 - 2 * area_under_lorenz

gini = gini_coefficient(final_wealth)
print(f"Gini coefficient of simulated wealth distribution: {gini:.4f}")
print(f"\nFor reference:")
print(f"  Perfect equality:     G = 0")
print(f"  Sweden (wealth):      G ≈ 0.50")
print(f"  Germany (wealth):     G ≈ 0.67")
print(f"  United States (wealth): G ≈ 0.85")
print(f"  Perfect inequality:   G = 1")

### Comparative Statics: How Parameters Shape Inequality

Now we explore how changing the model's parameters affects the wealth distribution. This is the computational equivalent of Piketty's historical analysis: what forces drive inequality up or down?

We compare three scenarios:
1. **Baseline**: $r = 0.02$, income spread $[0.5, 1.0, 2.0]$
2. **Higher interest rate**: $r = 0.05$ (capital earns more — Piketty's $r > g$)
3. **Higher income volatility**: income spread $[0.2, 1.0, 3.0]$ (wider gap between low and high)

In [ ]:
# Scenario 1: Baseline (already computed)
gini_baseline = gini

# Scenario 2: Higher interest rate
wealth_high_r = simulate_wealth(income_paths, r=0.05, alpha=0.9, w0=1.0)
gini_high_r = gini_coefficient(wealth_high_r[:, -1])

# Scenario 3: Higher income volatility
income_levels_volatile = np.array([0.2, 1.0, 3.0])
income_paths_volatile = income_levels_volatile[state_paths]
wealth_volatile = simulate_wealth(income_paths_volatile, r=0.02, alpha=0.9, w0=1.0)
gini_volatile = gini_coefficient(wealth_volatile[:, -1])

print("=== Comparative Statics ===")
print(f"{'Scenario':<30} {'Gini':>8} {'Mean Wealth':>14} {'Median Wealth':>14}")
print("-" * 68)
print(f"{'Baseline (r=0.02)':<30} {gini_baseline:>8.4f} {wealth[:,-1].mean():>14.2f} {np.median(wealth[:,-1]):>14.2f}")
print(f"{'Higher r (r=0.05)':<30} {gini_high_r:>8.4f} {wealth_high_r[:,-1].mean():>14.2f} {np.median(wealth_high_r[:,-1]):>14.2f}")
print(f"{'Higher volatility':<30} {gini_volatile:>8.4f} {wealth_volatile[:,-1].mean():>14.2f} {np.median(wealth_volatile[:,-1]):>14.2f}")

In [ ]:
# Plot all three distributions and their Lorenz curves
scenarios = [
    ('Baseline (r=0.02)', wealth[:, -1], 'steelblue'),
    ('Higher r (r=0.05)', wealth_high_r[:, -1], 'coral'),
    ('Higher volatility', wealth_volatile[:, -1], 'seagreen'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left panel: wealth distributions
for label, w, color in scenarios:
    axes[0].hist(w, bins=60, density=True, alpha=0.4, color=color, edgecolor='white', label=label)
axes[0].set_xlabel('Wealth', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('Wealth Distributions under Different Parameters', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right panel: Lorenz curves
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Perfect equality')
for label, w, color in scenarios:
    ps, ws = lorenz_curve(w)
    g = gini_coefficient(w)
    axes[1].plot(ps, ws, color=color, linewidth=2.5, label=f'{label} (Gini={g:.3f})')

axes[1].set_xlabel('Cumulative population share', fontsize=12)
axes[1].set_ylabel('Cumulative wealth share', fontsize=12)
axes[1].set_title('Lorenz Curves', fontsize=13)
axes[1].legend(fontsize=10, loc='upper left')
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key findings:")
print("  - A higher interest rate increases inequality (the rich earn more on their savings).")
print("  - Greater income volatility increases inequality (bigger shocks, harder to insure).")
print("  - Both push the Lorenz curve further from the line of equality.")

### How the Gini Evolves Over Time

In [ ]:
# Track the Gini coefficient over time for the baseline model
gini_over_time = []
time_points = range(1, n_periods, 5)

for t in time_points:
    gini_over_time.append(gini_coefficient(wealth[:, t]))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(time_points), gini_over_time, color='steelblue', linewidth=2.5)
ax.set_xlabel('Period', fontsize=12)
ax.set_ylabel('Gini Coefficient', fontsize=12)
ax.set_title('Evolution of the Gini Coefficient Over Time', fontsize=13)
ax.grid(True, alpha=0.3)
ax.axhline(gini_over_time[-1], color='red', linestyle='--', alpha=0.5,
           label=f'Converged value ≈ {gini_over_time[-1]:.3f}')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("All agents start with the same wealth (w0 = 1), so Gini starts near 0.")
print("As idiosyncratic shocks accumulate, inequality rises and converges to a steady state.")
print("This is the Aiyagari insight: inequality is endogenous, emerging from luck alone.")

---

## Part III — Exercises

### Exercise 1 — The Effect of Income Mobility

The transition matrix $P$ determines how persistent income shocks are. High persistence (staying in the same state for many periods) means income luck has lasting effects on wealth. Low persistence (frequent switching between states) means shocks wash out quickly.

**(a)** Create a **high-persistence** transition matrix where each state has a 0.8 probability of remaining and transitions are split equally between the other two states. Simulate 5,000 agents for 200 periods with baseline parameters ($r = 0.02$, $\alpha = 0.9$). Compute the Gini coefficient of the final wealth distribution.

**(b)** Create a **low-persistence** transition matrix where each state has a 0.4 probability of remaining. Again, split the remaining probability equally between the other two states. Simulate and compute the Gini.

**(c)** Plot the Lorenz curves for the high-persistence, low-persistence, and baseline cases on a single figure. Which has the most inequality? Which has the least? Explain intuitively why persistence matters.

**(d)** Compute the Gini coefficient for persistence values of 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, and 0.9 (where persistence is the diagonal element of the transition matrix). Plot Gini against persistence. What is the relationship?

In [ ]:
# Your answer here


### Exercise 2 — Progressive Taxation

One of the key policy tools against inequality is **progressive taxation**: taxing higher incomes at higher rates and redistributing the revenue.

**(a)** Implement a progressive tax-and-transfer system. Define a tax function that taxes income above a threshold at a higher rate:

$$\text{tax}(y) = \begin{cases} 0.1 \cdot y & \text{if } y \leq 1.0 \\ 0.1 + 0.4 \cdot (y - 1.0) & \text{if } y > 1.0 \end{cases}$$

Compute total tax revenue per period and redistribute it equally as a lump-sum transfer to all agents. The post-tax-and-transfer income is $y_t^{\text{net}} = y_t - \text{tax}(y_t) + \text{transfer}$. Simulate wealth accumulation using $y_t^{\text{net}}$ instead of $y_t$.

**(b)** Compare the Gini coefficient with and without the tax system. Plot both Lorenz curves.

**(c)** Experiment with a more aggressive progressive tax: raise the top rate from 0.4 to 0.6. How much further does the Gini fall? Is there a point of diminishing returns?

**(d)** Plot the wealth distributions (histograms) for the no-tax, moderate-tax, and high-tax scenarios on a single figure. In 2-3 sentences, discuss the trade-off between equality and the total wealth generated.

In [ ]:
# Your answer here


### Exercise 3 — Piketty's r > g

Piketty argued that when the return on capital ($r$) exceeds the growth rate ($g$), wealth inequality tends to increase without bound. Let's test this computationally.

**(a)** Modify the wealth accumulation equation to include economic growth. Income grows at rate $g$ per period: $y_t = y_0 \cdot (1 + g)^t$ (multiply the Markov chain income by a growth factor). Simulate with $r = 0.04$ and $g = 0.02$ (so $r > g$). Track the Gini over time for 500 periods.

**(b)** Now simulate with $r = 0.02$ and $g = 0.04$ (so $r < g$). Track the Gini over time. Compare the two trajectories on a single plot.

**(c)** For the $r > g$ case, compute the wealth share of the top 10% over time. Does it stabilise or keep growing? Plot it.

**(d)** Piketty argued that the mid-twentieth century was exceptional because wars and progressive taxation temporarily pushed the effective $r$ below $g$. Simulate a scenario where $r > g$ for the first 200 periods, then $r < g$ for periods 200-350 (the "Great Compression"), then $r > g$ again for periods 350-500. Plot the Gini over all 500 periods. Does the pattern resemble the historical U-shape of inequality?

In [ ]:
# Your answer here


---

## Part IV — Quiz

### Conceptual Questions

**Q1.** Piketty's central inequality, $r > g$, states that when the return on capital exceeds the growth rate:

(a) The economy enters recession  
(b) Inherited wealth grows faster than earned income, increasing inequality  
(c) Inflation accelerates  
(d) Government debt becomes unsustainable

**Q2.** A Gini coefficient of 0 means:

(a) Everyone has zero wealth  
(b) One person has all the wealth  
(c) Wealth is perfectly equally distributed  
(d) The economy is in autarky

**Q3.** The Lorenz curve plots:

(a) Income over time  
(b) Cumulative wealth share against cumulative population share  
(c) GDP growth against inflation  
(d) Supply against demand

**Q4.** In the Aiyagari model, inequality arises because:

(a) Agents have different preferences and abilities  
(b) Some agents are inherently smarter than others  
(c) Identical agents face different idiosyncratic income shocks and cannot perfectly insure  
(d) The government redistributes from poor to rich

**Q5.** Piketty's *Capital in the Twenty-First Century* argued that the mid-twentieth-century decline in inequality was:

(a) The natural and permanent tendency of capitalism  
(b) An anomaly caused by wars, depression, and progressive taxation  
(c) The result of superior economic theory  
(d) A statistical illusion caused by measurement error

### Computational Questions

**Q6.** If the Lorenz curve passes through the point (0.5, 0.2), this means:

(a) The poorest 50% of the population holds 20% of total wealth  
(b) The richest 20% holds 50% of total wealth  
(c) Average wealth is 0.2  
(d) The Gini coefficient is 0.5

**Q7.** A Gini coefficient of 0.85 (approximately the US wealth Gini) indicates:

(a) Moderate inequality  
(b) Nearly perfect equality  
(c) Very high inequality — the Lorenz curve bows far below the line of equality  
(d) That 85% of people have equal wealth

**Q8.** In a Markov chain with transition matrix $P$, the stationary distribution $\pi$ satisfies:

(a) $\pi = P$  
(b) $\pi P = \pi$ (it is unchanged by one application of the transition matrix)  
(c) $\pi + P = 1$  
(d) $\pi P = 0$

**Q9.** Increasing the persistence of the income Markov chain (higher diagonal elements in $P$) will:

(a) Decrease inequality because income is more stable  
(b) Increase inequality because long runs of bad (or good) luck have lasting effects on wealth  
(c) Have no effect on the wealth distribution  
(d) Make the Gini coefficient exactly 1

**Q10.** In the wealth accumulation equation $w_{t+1} = (1+r) \cdot w_t + y_t - c_t$, the borrowing constraint $w_t \geq 0$ matters because:

(a) It prevents agents from saving  
(b) It ensures all agents have the same wealth  
(c) It prevents agents from perfectly insuring against bad shocks, which generates inequality  
(d) It makes the interest rate irrelevant

### Quiz Answers

<details>
<summary>Click to reveal answers</summary>

**Q1.** (b) When $r > g$, those who already own capital see their wealth grow faster than the overall economy, meaning inherited wealth outpaces earned income and inequality widens.

**Q2.** (c) A Gini of 0 means the Lorenz curve coincides with the 45-degree line — every person has exactly the same wealth.

**Q3.** (b) The Lorenz curve plots the cumulative share of total wealth (y-axis) held by the cumulative share of the population (x-axis), sorted from poorest to richest.

**Q4.** (c) The Aiyagari model's key insight is that identical agents facing uninsurable idiosyncratic shocks generate an unequal wealth distribution as an emergent property — inequality without innate differences.

**Q5.** (b) Piketty argued that the "Great Compression" of 1930-1975 was a historical anomaly driven by the destruction of capital in two world wars, the Great Depression, and deliberate policy choices (progressive taxation, welfare states), not the natural tendency of capitalism.

**Q6.** (a) The Lorenz curve point (0.5, 0.2) means that when the population is sorted from poorest to richest, the bottom half collectively holds only 20% of total wealth.

**Q7.** (c) A Gini of 0.85 is very close to 1 (perfect inequality). The Lorenz curve bows dramatically below the diagonal, indicating extreme concentration of wealth.

**Q8.** (b) The stationary distribution $\pi$ is a fixed point of the transition: applying $P$ once leaves it unchanged. It represents the long-run fraction of time spent in each state.

**Q9.** (b) Higher persistence means agents who draw a low income state stay there longer, depleting their savings, while high-income agents accumulate more. This amplifies wealth differences and increases the Gini.

**Q10.** (c) Without the borrowing constraint, agents could borrow against future good income to smooth consumption perfectly. The constraint prevents this, so agents hit by bad shocks cannot offset them by borrowing — this is the market incompleteness that generates inequality in the Aiyagari model.

</details>

---

## Further Reading

- Piketty, T. *Capital in the Twenty-First Century* (2014), especially the Introduction and Chapters 1, 7-12.
- Aiyagari, S.R. "Uninsured Idiosyncratic Risk and Aggregate Saving," *Quarterly Journal of Economics* (1994).
- Deaton, A. *The Great Escape: Health, Wealth, and the Origins of Inequality* (2013).
- Piketty, T. and Saez, E. "Income Inequality in the United States, 1913-1998," *Quarterly Journal of Economics* (2003).
- Kuznets, S. "Economic Growth and Income Inequality," *American Economic Review* (1955).

---

*Next week: we move from the distribution of wealth within countries to the dynamics of growth between them — convergence, divergence, and the puzzle of why some nations are rich and others poor.*